# 04 — Build Your Own MCP Client

**Goal:** Build a client that connects to your MCP server, discovers tools, and calls them.

## What the client does

1. Launch the server as a subprocess
2. Perform the initialization handshake
3. Discover available tools (`tools/list`)
4. Call tools on demand (`tools/call`)
5. Shut down cleanly

This is exactly what sbot Layer 9 will do — connect to external MCP servers and use their tools.

## Exercise 4.1 — Build the MCP client class

Fill in the TODOs to create a working client.

In [ ]:
import asyncio
import json

class MiniMCPClient:
    """Minimal MCP client — connects to a server via stdio."""
    
    def __init__(self, server_command: list[str]):
        self.server_command = server_command
        self.proc = None
        self._next_id = 1
        self.server_info = None
        self.server_capabilities = None
        self.tools = []  # populated after list_tools()
    
    async def start(self):
        """Launch server subprocess."""
        self.proc = await asyncio.create_subprocess_exec(
            *self.server_command,
            stdin=asyncio.subprocess.PIPE,
            stdout=asyncio.subprocess.PIPE,
            stderr=asyncio.subprocess.PIPE,
        )
    
    async def _send_request(self, method: str, params: dict | None = None) -> dict:
        """Send a JSON-RPC request and wait for the response."""
        msg = {"jsonrpc": "2.0", "id": self._next_id, "method": method}
        if params:
            msg["params"] = params
        self._next_id += 1
        
        line = json.dumps(msg) + "\n"
        self.proc.stdin.write(line.encode())
        await self.proc.stdin.drain()
        
        resp_line = await self.proc.stdout.readline()
        resp = json.loads(resp_line.decode())
        
        # Check for JSON-RPC error
        if "error" in resp:
            raise RuntimeError(f"RPC error: {resp['error']['message']}")
        
        return resp["result"]
    
    async def _send_notification(self, method: str, params: dict | None = None):
        """Send a JSON-RPC notification (no response expected)."""
        msg = {"jsonrpc": "2.0", "method": method}
        if params:
            msg["params"] = params
        line = json.dumps(msg) + "\n"
        self.proc.stdin.write(line.encode())
        await self.proc.stdin.drain()
    
    async def initialize(self):
        """Perform the MCP initialization handshake."""
        # TODO:
        # 1. Send "initialize" request with:
        #    - protocolVersion: "2024-11-05"
        #    - capabilities: {} (we don't support sampling yet)
        #    - clientInfo: {"name": "mini-client", "version": "1.0.0"}
        # 2. Store server_info and server_capabilities from response
        # 3. Send "notifications/initialized" notification
        pass
    
    async def list_tools(self) -> list[dict]:
        """Discover available tools from the server."""
        # TODO:
        # 1. Send "tools/list" request
        # 2. Store tools in self.tools
        # 3. Return the list
        pass
    
    async def call_tool(self, name: str, arguments: dict) -> dict:
        """Call a tool and return the result."""
        # TODO:
        # 1. Send "tools/call" request with params: {"name": ..., "arguments": ...}
        # 2. Return the result (which has "content" and "isError" fields)
        pass
    
    async def shutdown(self):
        """Shut down the server."""
        if self.proc:
            self.proc.stdin.close()
            await self.proc.wait()

print("MiniMCPClient defined. Run the next cell to test it.")

## Exercise 4.2 — Test the full client-server flow

Connect to your `mini_mcp_server.py` from notebook 03.

In [ ]:
async def test_full_flow():
    client = MiniMCPClient(["python3", "mini_mcp_server.py"])
    
    # 1. Start server
    await client.start()
    print("Server started")
    
    # 2. Initialize
    await client.initialize()
    print(f"Connected to: {client.server_info}")
    print(f"Capabilities: {client.server_capabilities}")
    
    # 3. Discover tools
    tools = await client.list_tools()
    print(f"\nAvailable tools ({len(tools)}):")
    for t in tools:
        print(f"  - {t['name']}: {t['description']}")
    
    # 4. Call tools
    print("\n--- Calling tools ---")
    
    result = await client.call_tool("add", {"a": 17, "b": 25})
    print(f"add(17, 25) = {result['content'][0]['text']}")
    assert result["content"][0]["text"] == "42"
    
    result = await client.call_tool("get_weather", {"city": "Ho Chi Minh City"})
    print(f"weather: {result['content'][0]['text']}")
    
    result = await client.call_tool("list_files", {"path": "."})
    files = result['content'][0]['text'].split('\n')
    print(f"files in '.': {files[:5]}...")
    
    # 5. Shutdown
    await client.shutdown()
    print("\nServer shut down cleanly")
    print("\n✓ Full MCP client-server flow working!")

await test_full_flow()

## Exercise 4.3 — Simulate an LLM using MCP tools

This is the key insight: in a real AI agent, the **LLM decides** which tool to call. The MCP client just executes what the LLM requests.

Let's simulate this flow:
1. Get tool list from MCP server
2. Show tools to "LLM" (we'll fake the LLM decision)
3. Execute the tool call the LLM chose
4. Return result to the LLM

In [ ]:
async def simulate_agent_with_mcp():
    """Simulates how sbot would use MCP tools."""
    
    # --- Connect to MCP server ---
    client = MiniMCPClient(["python3", "mini_mcp_server.py"])
    await client.start()
    await client.initialize()
    tools = await client.list_tools()
    
    # --- Convert MCP tools to "LLM tool format" ---
    # In real sbot: these would become LangChain tools via bind_tools()
    print("Tools available to LLM:")
    for t in tools:
        print(f"  {t['name']}: {t['description']}")
        print(f"    params: {json.dumps(t['inputSchema']['properties'])}")
    
    # --- Simulate user question ---
    user_message = "What's the weather in Hanoi?"
    print(f"\nUser: {user_message}")
    
    # --- Simulate LLM deciding to call a tool ---
    # (In reality, the LLM would analyze the user message + tool descriptions)
    llm_tool_call = {
        "name": "get_weather",
        "arguments": {"city": "Hanoi"}
    }
    print(f"LLM decides: call {llm_tool_call['name']}({llm_tool_call['arguments']})")
    
    # --- Execute via MCP ---
    result = await client.call_tool(llm_tool_call["name"], llm_tool_call["arguments"])
    tool_output = result["content"][0]["text"]
    print(f"Tool result: {tool_output}")
    
    # --- Feed result back to LLM ---
    # (In reality, this goes back as a ToolMessage in the conversation)
    print(f"\nLLM (with tool result): The weather in Hanoi is {tool_output}")
    
    await client.shutdown()
    print("\n✓ Agent + MCP flow complete!")

await simulate_agent_with_mcp()

### Question 4.3

In this flow, who decides WHICH tool to call — the MCP client or the LLM?

And who actually RUNS the tool — the MCP client or the MCP server?

This separation is the core design insight of MCP:
- **LLM** (in the host) decides what to do
- **MCP server** does the actual work
- **MCP client** is just the messenger between them

*Your answer:*


## Exercise 4.4 — YOUR TURN: Add a new tool end-to-end

Add a `read_file` tool to `mini_mcp_server.py`:
1. Edit `mini_mcp_server.py` — add tool definition + handler
2. Test it with the client below

The tool should:
- Name: `read_file`
- Params: `path` (string, required)
- Returns: file contents as text
- On error (file not found): return `isError: true` with error message

In [ ]:
# After you've edited mini_mcp_server.py, run this to test

async def test_read_file_tool():
    client = MiniMCPClient(["python3", "mini_mcp_server.py"])
    await client.start()
    await client.initialize()
    
    # Should now have 4 tools
    tools = await client.list_tools()
    tool_names = [t["name"] for t in tools]
    print(f"Tools: {tool_names}")
    assert "read_file" in tool_names, "Add read_file tool to mini_mcp_server.py first!"
    
    # Read this notebook
    result = await client.call_tool("read_file", {"path": "mini_mcp_server.py"})
    content = result["content"][0]["text"]
    print(f"Read {len(content)} chars from mini_mcp_server.py")
    print(f"First line: {content.split(chr(10))[0]}")
    assert result["isError"] == False
    
    # Try reading nonexistent file
    result = await client.call_tool("read_file", {"path": "/nonexistent/file.txt"})
    print(f"\nNonexistent file: isError={result['isError']}")
    print(f"Error: {result['content'][0]['text']}")
    assert result["isError"] == True
    
    await client.shutdown()
    print("\n✓ read_file tool works!")

await test_read_file_tool()

## Summary

You now have a complete DIY MCP implementation:

```
mini_mcp_server.py          Your MCP server (tools + stdio)
MiniMCPClient               Your MCP client (subprocess + JSON-RPC)
simulate_agent_with_mcp()   Shows how an LLM agent uses MCP
```

This is exactly the pattern sbot Layer 9 will use:
1. Config says "connect to these MCP servers"
2. sbot launches each server as a subprocess
3. Discovers their tools via `tools/list`
4. Merges them into the LangChain tool registry
5. When the LLM calls an MCP tool → forwards to the right server via `tools/call`

**Next notebook:** See how the official `mcp` Python SDK does all this with much less code